In [1]:
# imports
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [11]:
# data

#file_path = r"C:\Users\mrang\Downloads\master_0_dataset.csv"
file_path = "../data/master_rewrites_dataset.csv"
data = pd.read_csv(file_path)
data.head()

,id,text,label,generation_type,source,source_id
0,0,Title: Waqar Younis Resigns as Pakistan Coach...,ai,ai_rewrite,news,news_538
1,1,"Kolkata, India - West Indies' mercenary squad...",ai,ai_rewrite,news,news_416
2,2,"on. A female lies on her back, calling like cr...",human,human,essays_texts.csv,essays_1388_chunk3
3,3,LONDON: Joe Root was named Test Player of the ...,human,human,articles.csv,articles_1754
4,4,strong>ISLAMABAD: Guinness Book of World Recor...,human,human,articles.csv,articles_1994


## Data Cleaning

In [12]:
# normalize text/labels
data['text'] = data['text'].astype(str)
data['label'] = data['label'].str.lower().str.strip()

# numberic labels
label_map = {'human': 0, 'ai': 1}
data['label_num'] = data['label'].map(label_map)

# drop bad rows
data = data.dropna(subset=["text", "label_num"])
data = data[data["text"].str.strip() != ""]

In [13]:
# features + labels
X_text = data['text']
y = data['label_num']

## TF-IDF + K-Fold

In [14]:
# tf-idf features pipeline

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words='english',
        max_features=5000
    )),
    ("nb", MultinomialNB()) # naive bayes classifier
])

In [15]:
# stratified 10 fold cross validation 
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

accuracy_scores = cross_val_score(
    pipeline,
    X_text,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Accuracy scores for each fold:")
print(accuracy_scores)

print("\nMean accuracy:", accuracy_scores.mean())

Accuracy scores for each fold:
[0.68333333 0.675      0.71666667 0.63333333 0.65       0.775
 0.66666667 0.69166667 0.71666667 0.7       ]

Mean accuracy: 0.6908333333333334


In [16]:
# predictions + report
y_pred = cross_val_predict(
    pipeline,
    X_text,
    y,
    cv=cv
)

print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=['human', 'ai']))


Classification Report:
              precision    recall  f1-score   support

       human       0.68      0.71      0.70       600
          ai       0.70      0.67      0.68       600

    accuracy                           0.69      1200
   macro avg       0.69      0.69      0.69      1200
weighted avg       0.69      0.69      0.69      1200



## Group K-Fold

Here i attempted a different method and used grouped cross validation (GroupKfold) so the data was split by source instead of randomly, removed the noticeable difference in data and this is where the accuracy dropped, so the model was relying on those differences instead of learning the text.

In [17]:
# source as group
groups = data['source']

group_cv = GroupKFold(n_splits=3)

y_pred_group = cross_val_predict(
    pipeline,
    X_text,
    y,
    cv=group_cv,
    groups=groups
)

print("\nGroupKFold Results (by source):")
print(classification_report(y, y_pred_group, target_names=['human', 'ai']))


GroupKFold Results (by source):
              precision    recall  f1-score   support

       human       0.26      0.29      0.28       600
          ai       0.20      0.17      0.18       600

    accuracy                           0.23      1200
   macro avg       0.23      0.23      0.23      1200
weighted avg       0.23      0.23      0.23      1200



## Confusion Matrix

In [18]:
print("\nConfusion Matrix (Stratified):")
print(confusion_matrix(y, y_pred))

print("\nConfusion Matrix (GroupKFold):")
print(confusion_matrix(y, y_pred_group))


Confusion Matrix (Stratified):
[[429 171]
 [200 400]]

Confusion Matrix (GroupKFold):
[[175 425]
 [497 103]]


## Check top 30 features model uses to identify AI

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer

data['target'] = data['label'].map({'human': 0, 'ai': 1})

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X = vectorizer.fit_transform(data['text'])
y = data['target']

# predict human v ai w random forest
clf = RandomForestClassifier(n_estimators=200, random_state=42)
scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')

print("validation accuracy:", scores.mean())

# top leaking features
clf.fit(X, y)
feature_names = vectorizer.get_feature_names_out()
importances = clf.feature_importances_
top_features = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:30]

print("Top 30 features:")
for feat, score in top_features[:30]:
    print(f"{feat:40} {score:.5f}")

validation accuracy: 0.8458333333333334
Top 30 features:
said                                     0.01606
strong                                   0.01072
truly                                    0.01039
despite                                  0.00952
following                                0.00922
expressed                                0.00914
let tell                                 0.00889
good                                     0.00787
significant                              0.00761
let                                      0.00737
stated                                   0.00606
opted                                    0.00593
great                                    0.00565
really                                   0.00526
perspective                              0.00505
experience                               0.00489
spot                                     0.00472
potentially                              0.00444
unique                                   0.00416
tell        